# El kernel de Triton fuera de la H100

Todo el resultado de este kernel se midió en H100 alquiladas por hora. Eso
deja dos cosas sin saber.

**¿Compila en otra parte?** Triton exige capacidad de cómputo **7.0 o más**.
La **T4** de Kaggle es 7.5 y debería funcionar; la **P100** es 6.0 y lo más
probable es que se niegue. Si se niega, eso *es* un resultado — elige T4.

**¿La ganancia es una propiedad de la H100?** La H100 tiene 132 SMs y una L2
enorme. Un *stencil* que vive en registros podría comportarse muy distinto en
una T4 de 40 SMs, y esa es justo la pregunta.

Orden del cuadernillo, y no se cronometra nada antes de que la física
coincida:

1. entorno y capacidad de cómputo
2. **exactitud** — el *stencil* contra `sim_core.substep` con la **misma**
   aleatoriedad: debe dar **cero** espines distintos
3. **física** — las dos cadenas a equilibrio, cada una con su propio RNG
4. **ritmo del paso**, y la regla del ancho puntuada por separado para cada
   motor
5. **escalera corta**, para que la cifra citada sea el *anneal* completo y no
   un paso suelto

> Acelerador: **GPU T4 x2** (usa una sola). Tiempo estimado: 10-15 min,
> casi todo compilación.


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi",
  "--query-gpu=name,memory.total,clocks.max.sm,driver_version",
  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

## Instalación

`jax-triton` trae su propio `triton` y arrastra una versión de `jax`
compatible. En la H100 la combinación que funcionó fue
`jax 0.10.2 / triton 3.8.0 / jax_triton 0.3.1`; aquí se deja que pip resuelva
y se imprime lo que quedó, porque una discrepancia de versiones es la causa
más probable de que esto no arranque.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "jax[cuda12]", "jax-triton"], check=False)
for m in [k for k in list(sys.modules) if k.split(".")[0] in
          ("jax", "jaxlib", "triton", "jax_triton")]:
    del sys.modules[m]
import jax
print("jax", jax.__version__, jax.devices())
try:
    import triton, jax_triton as jt
    print("triton", triton.__version__,
          "| jax_triton", getattr(jt, "__version__", "?"))
except Exception as e:
    print("FALTA:", type(e).__name__, e)

In [ ]:
# sim_core.py -- byte-identical to the repository copy. sha256 840b68448e8391ae9b07aa7a1161cb9d36cef67667489288756d3c044a66d058
import hashlib
_src = '"""Monte Carlo engine for the nanodisk, faithful to the project\'s reference notebook.\n\nTaken from `00_benchmark_mc_observables.ipynb` -- the same checkerboard, the same\nuniform-sphere proposal, and in particular the same DMI: **in-plane bonds only\n(x and y), with a minus sign**. That is interfacial DMI, the physics of a thin\nfilm whose DM vector comes from an interface, and it is not interchangeable with\nthe all-axes form: on the same configuration the total energy differs by 50%.\n\nTwo terms the reference notebook does not carry are added here because the\ntutor\'s dataset varies them:\n\n- `KanS`, the surface anisotropy, applied where a site has fewer than six\n  in-disk nearest neighbours (the coordination test used in the group\'s\n  multi-GPU notebooks).\n- `Kan2`, read as the SECOND-ORDER constant of the cubic anisotropy expansion,\n\n      E_an = (Kan1 / So^4)(sx^2 sy^2 + sx^2 sz^2 + sy^2 sz^2)\n           + (Kan2 / So^6)(sx^2 sy^2 sz^2)\n\n  which is the textbook K1/K2 series and matches the naming. It is a single-site\n  term, so unlike a second-neighbour coupling it leaves the checkerboard exact.\n"""\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom jax import random\n\n\ndef build_geometry(Rd: float = 18.30, L: int = 5):\n    """Disk mask, checkerboard sublattices and the surface mask."""\n    L2 = int(Rd) + 1\n    N = 2 * L2 + 1\n    x = np.arange(-L2, L2 + 1)\n    xg, yg = np.meshgrid(x, x, indexing="xy")\n    disk2d = (xg ** 2 + yg ** 2) <= Rd ** 2\n    disco = np.broadcast_to(disk2d, (L, N, N)).copy()\n\n    zc, yc, xc = np.indices((L, N, N))\n    checker = (xc + yc + zc) % 2\n    even = (checker == 0) & disco\n    odd = (checker == 1) & disco\n\n    # Surface = fewer than six in-disk nearest neighbours. x and y wrap, z does\n    # not, so the top and bottom layers are surface everywhere.\n    surface = np.zeros_like(disco, dtype=bool)\n    for z in range(L):\n        for y in range(N):\n            for xx in range(N):\n                if not disco[z, y, xx]:\n                    continue\n                n = 0\n                for dz, dy, dx in ((0, 0, 1), (0, 0, -1), (0, 1, 0),\n                                   (0, -1, 0), (1, 0, 0), (-1, 0, 0)):\n                    nz, ny, nx = z + dz, (y + dy) % N, (xx + dx) % N\n                    if 0 <= nz < L and disco[nz, ny, nx]:\n                        n += 1\n                if n < 6:\n                    surface[z, y, xx] = True\n    return dict(L=L, N=N, Rd=Rd, disco=disco, even=even, odd=odd,\n                surface=surface, disk2d=disk2d, n_active=int(disco.sum()),\n                n_surface=int(surface.sum()))\n\n\ndef build_engine(geo, So=1.0):\n    """Returns (substep, energy, magnetization, topological_charge, anneal)."""\n    L, N = geo["L"], geo["N"]\n    even_j = jnp.asarray(geo["even"][None, ..., None])\n    odd_j = jnp.asarray(geo["odd"][None, ..., None])\n    disco_j = jnp.asarray(geo["disco"][None, ..., None])\n    surf_j = jnp.asarray(geo["surface"][None, ...])\n    disk2d_j = jnp.asarray(geo["disk2d"])\n    n_act = geo["n_active"]\n\n    def _an(s, Kan1, Kan2, K1_eff):\n        """Cubic anisotropy: the K1 invariant plus the K2 (sixth-order) one."""\n        sx, sy, sz = s[..., 0], s[..., 1], s[..., 2]\n        e = (K1_eff / So ** 4) * ((sx * sy) ** 2 + (sx * sz) ** 2 + (sy * sz) ** 2)\n        return e + (Kan2 / So ** 6) * (sx * sy * sz) ** 2\n\n    def substep(spins, key, beta, mask, p, K1_eff):\n        S_xp = jnp.roll(spins, -1, 3); S_xm = jnp.roll(spins, 1, 3)\n        S_yp = jnp.roll(spins, -1, 2); S_ym = jnp.roll(spins, 1, 2)\n        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)\n        S_zm = jnp.roll(spins, 1, 1).at[:, 0].set(0.0)\n\n        kt, ka = random.split(key)\n        tr = random.normal(kt, spins.shape)\n        tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So\n        tr = jnp.where(mask, tr, spins)\n        dS = tr - spins\n\n        sv = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm\n        dE = -p["Jex"] * jnp.sum(dS * sv, axis=-1)\n        dE = dE + _an(tr, p["Kan1"], p["Kan2"], K1_eff) - _an(spins, p["Kan1"], p["Kan2"], K1_eff)\n        # Interfacial DMI: in-plane bonds only, minus sign. See module docstring.\n        dE = dE - p["KDM"] * (dS[..., 1] * (S_xp[..., 2] - S_xm[..., 2])\n                              - dS[..., 2] * (S_xp[..., 1] - S_xm[..., 1])\n                              + dS[..., 2] * (S_yp[..., 0] - S_ym[..., 0])\n                              - dS[..., 0] * (S_yp[..., 2] - S_ym[..., 2]))\n        dE = dE - p["Hex"] * (dS[..., 0] * jnp.cos(p["g"]) + dS[..., 2] * jnp.sin(p["g"]))\n\n        r = random.uniform(ka, dE.shape)\n        acc = ((dE < 0) | (r < jnp.exp(-dE * beta)))[..., None] & mask\n        return jnp.where(acc, tr, spins)\n\n    def energy(spins, p, K1_eff):\n        """Total energy; each bond counted once via the +x, +y, +z shifts only."""\n        S_xp = jnp.roll(spins, -1, 3)\n        S_yp = jnp.roll(spins, -1, 2)\n        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)\n        E = -p["Jex"] * jnp.sum(spins * (S_xp + S_yp + S_zp), axis=-1)\n        sx, sy, sz = spins[..., 0], spins[..., 1], spins[..., 2]\n        E = E + _an(spins, p["Kan1"], p["Kan2"], K1_eff)\n        E = E - p["KDM"] * (sy * S_xp[..., 2] - sz * S_xp[..., 1]\n                            + sz * S_yp[..., 0] - sx * S_yp[..., 2])\n        E = E - p["Hex"] * (sx * jnp.cos(p["g"]) + sz * jnp.sin(p["g"]))\n        return jnp.sum(E * disco_j[..., 0], axis=(-3, -2, -1))\n\n    def magnetization(spins):\n        M = jnp.sum(spins * disco_j, axis=(1, 2, 3)) / n_act\n        return jnp.linalg.norm(M, axis=-1), M[:, 2]\n\n    def _omega(s1, s2, s3):\n        num = jnp.sum(s1 * jnp.cross(s2, s3), axis=-1)\n        den = (1.0 + jnp.sum(s1 * s2, -1) + jnp.sum(s2 * s3, -1) + jnp.sum(s3 * s1, -1))\n        return 2.0 * jnp.arctan2(num, den)\n\n    def topo_charge(spins):\n        """Berg-Luscher charge of every layer: quantised, unlike a finite difference."""\n        s00 = spins[..., :-1, :-1, :]; s10 = spins[..., :-1, 1:, :]\n        s01 = spins[..., 1:, :-1, :];  s11 = spins[..., 1:, 1:, :]\n        valid = (disk2d_j[:-1, :-1] & disk2d_j[:-1, 1:]\n                 & disk2d_j[1:, :-1] & disk2d_j[1:, 1:])\n        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)\n        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (4.0 * jnp.pi)\n\n    def anneal(spins, key, betas, p, n_therm, n_meas):\n        K1_eff = jnp.where(surf_j, p["KanS"], p["Kan1"])\n\n        def per_T(carry, beta):\n            s, k = carry\n\n            def therm(_, v):\n                s, k = v\n                k, k1, k2 = random.split(k, 3)\n                s = substep(s, k1, beta, even_j, p, K1_eff)\n                s = substep(s, k2, beta, odd_j, p, K1_eff)\n                return s, k\n            s, k = jax.lax.fori_loop(0, n_therm, therm, (s, k))\n\n            b = s.shape[0]\n            z0 = jnp.zeros(b)\n\n            # C_v and chi are variances, and accumulating raw E and E^2\n            # destroys them in float32. At T = 0.2 this run has <E> ~ -1.7e4, so\n            # <E^2> ~ 2.9e8, where one float32 ulp is ~34 -- larger than the\n            # variance itself. <E^2> - <E>^2 then comes out NEGATIVE and C_v\n            # reads -437: not a small error but pure cancellation.\n            #\n            # Measuring the fluctuation about a reference taken once, before the\n            # loop, keeps every accumulated quantity at the scale of the\n            # fluctuation rather than of the total. A constant shift leaves the\n            # variance unchanged, so this is exact. Both the centred moments and\n            # the reference are returned, and `reduce_obs` recombines them.\n            E_ref = energy(s, p, K1_eff)\n            M_ref, _ = magnetization(s)\n\n            def meas(_, v):\n                s, k, sE, sE2, sM, sM2, sMz, sQ = v\n                k, k1, k2 = random.split(k, 3)\n                s = substep(s, k1, beta, even_j, p, K1_eff)\n                s = substep(s, k2, beta, odd_j, p, K1_eff)\n                dE = energy(s, p, K1_eff) - E_ref\n                M, Mz = magnetization(s)\n                dM = M - M_ref\n                Q = topo_charge(s)[:, L // 2]\n                return (s, k, sE + dE, sE2 + dE * dE,\n                        sM + dM, sM2 + dM * dM, sMz + Mz, sQ + Q)\n\n            s, k, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(\n                0, n_meas, meas, (s, k, z0, z0, z0, z0, z0, z0))\n            obs = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            # One middle-layer snapshot per temperature, replica 0 only. The\n            # whole batch would be 30 x 100 x 5 x 39 x 39 x 3 floats (~2.7 GB);\n            # one replica\'s middle layer is 182 kB and is what the state-\n            # transition figure needs.\n            snap = s[0, L // 2, :, :, 2]\n            return (s, k), (obs, snap)\n\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (spins, key), betas)\n        return all_obs, s, snaps\n\n    return dict(substep=substep, energy=energy, magnetization=magnetization,\n                topo_charge=topo_charge, anneal=anneal)\n\n\ndef init_spins(key, geo, batch, So=1.0):\n    L, N = geo["L"], geo["N"]\n    x = random.normal(key, (batch, L, N, N, 3))\n    x = x / jnp.linalg.norm(x, axis=-1, keepdims=True) * So\n    return x * jnp.asarray(geo["disco"][None, ..., None])\n\n\ndef reduce_obs(obs, temps, n_spins, kB=0.086173404):\n    """Accumulators -> per-temperature scalars.\n\n    `lax.scan` stacks each step\'s output along a NEW LEADING axis, so `anneal`\n    returns `(n_T, 6, batch)` and the observable index is the SECOND axis. The\n    obvious `obs[i]` indexes temperatures instead, silently mixing the two --\n    and a test with six temperatures and six observables cannot tell the\n    difference, since both shapes read `(6, 6, batch)`.\n    """\n    obs = np.asarray(obs)\n    assert obs.ndim == 3 and obs.shape[1] == 8, (\n        f"expected (n_T, 8, batch) from lax.scan, got {obs.shape}")\n    dE, dE2, dM, dM2, Mz, Q, E_ref, M_ref = [obs[:, i] for i in range(8)]\n    T = np.asarray(temps)[:, None]\n    # Variances of the CENTRED quantities: both terms sit at the scale of the\n    # fluctuation, so there is nothing to cancel.\n    var_E = dE2 - dE ** 2\n    var_M = dM2 - dM ** 2\n    E = dE + E_ref\n    M = dM + M_ref\n    Cv = var_E / (kB * T ** 2 * n_spins)\n    chi = var_M / (kB * T)\n    return dict(T=np.asarray(temps),\n                E=E.mean(1) / n_spins, E_sd=E.std(1) / n_spins,\n                M=M.mean(1), M_sd=M.std(1),\n                Mz=Mz.mean(1), Cv=Cv.mean(1), chi=chi.mean(1),\n                Q=Q.mean(1), Q_sd=Q.std(1))\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "840b68448e8391ae9b07aa7a1161cb9d36cef67667489288756d3c044a66d058", "sim_core.py alterado"
open("sim_core.py", "w").write(_src)
print("sim_core.py  {} bytes  sha256 ok".format(len(_src)))


In [ ]:
# triton_halo.py -- byte-identical to the repository copy. sha256 1d36dd32e4bddfa299b91b89f305165b5598cae8c8ba4932ab9fdeeef1b4b862
import hashlib
_src = '"""A raw-Triton substep that keeps the neighbourhood in registers.\n\nWhy this exists\n---------------\nThe JAX substep builds six shifted copies of the lattice with `jnp.roll` and\nthen does arithmetic on them. HLO attributes 31.5% of the anneal to those six\nrolls alone. Three earlier attempts to remove them went through Pallas and all\nlost, by +15% to +31%, for one reason recorded in\n`the-cost-is-the-gather-not-the-lanes`: each lane computed its own neighbour\naddresses, so the compiler could no longer prove the accesses were contiguous\nand emitted scattered loads. 1521 sites x 7 neighbours x 3 components is about\n32k address computations per site-step, against seven coalesced array reads in\nthe JAX path.\n\nThis kernel is the version that was never writable in Pallas. The difference\nis not fewer loads -- it still issues 21 of them -- but their SHAPE:\n\n    Pallas gather   : one address per lane, computed per lane  -> scattered\n    this kernel     : one contiguous run per row, shifted      -> coalesced\n\nA shifted contiguous load is the same access pattern a roll produces, except\nthe shifted plane is never materialised. It lands in registers, the whole\nupdate happens there, and only the result is stored. So the six rolls\'\nround-trip through memory disappears while the coalescing survives. In Triton\nterms that is "the halo in SRAM": the overlapping shifted loads of adjacent\nblocks hit L1/L2, and the stencil never writes an intermediate.\n\nWhy the halo needs no coordination: the update is a checkerboard, so every\nneighbour of an updated site has the OPPOSITE parity and no block writes it\nduring this substep. A block can therefore read its halo straight from global\nmemory with no barrier -- which is the whole reason the checkerboard exists.\n\nLayout\n------\nProduction spins are `(B, L, N, N, 3)`, component-minor, so a run along x has\nstride 3. This kernel takes `(B, 3, L, N, N)`, component-planar, where a run\nalong x is stride 1 and a row load is one contiguous 39-float segment. The\ntranspose is paid once outside the step loop, so over thousands of steps it\ncosts nothing.\n\nTwo variants, for two different jobs\n------------------------------------\n`HAS_RNG=False` takes the trial spins and the Metropolis uniforms as inputs,\nso it can be compared against `sim_core.substep` on identical randomness --\nthat validates the stencil. `HAS_RNG=True` draws them in-kernel with Philox,\nwhich is the real candidate: it also absorbs the 17.1% of the anneal that\nHLO attributes to Threefry.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nimport triton\nimport triton.language as tl\n\n\n# ---------------------------------------------------------------- the kernel\n\n@triton.jit\ndef _substep_kernel(\n    sp_ptr,          # (B, 3, LZ, N, N)  f32   spins in\n    k1_ptr,          # (B, LZ, N, N)     f32   K1_eff, surface-dependent\n    par_ptr,         # (B, 6)            f32   Jex, Kan2, Hex, KDM, cos g, sin g\n    disk_ptr,        # (N, N)            f32   1.0 inside the disk\n    beta_ptr,        # (1,)              f32\n    step_ptr,        # (1,)              i32   mixes into the Philox seed\n    tr_ptr,          # (B, 3, LZ, N, N)  f32   trial spins   (HAS_RNG=False)\n    ru_ptr,          # (B, LZ, N, N)     f32   uniforms      (HAS_RNG=False)\n    out_ptr,         # (B, 3, LZ, N, N)  f32   spins out\n    de_ptr,          # (B, LZ, N, N)     f32   dE out        (WRITE_DE=True)\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    PARITY: tl.constexpr,     # 0 updates the even sublattice, 1 the odd\n    BY: tl.constexpr,         # rows of the block\n    BX: tl.constexpr,         # lanes along x, a power of two >= N\n    HAS_RNG: tl.constexpr,\n    WRITE_DE: tl.constexpr,\n    SO: tl.constexpr,\n):\n    b = tl.program_id(0)\n    z = tl.program_id(1)\n    ty = tl.program_id(2)\n\n    NN = N * N\n    sb = 3 * LZ * NN          # stride between replicas\n    sc = LZ * NN              # stride between components\n\n    # ---- the block, and its neighbour index vectors.\n    # These are VECTORS over the block, not per-lane addresses: `xc + 1` is a\n    # contiguous run shifted by one element, which is what keeps the loads\n    # coalesced. That is the single difference from the gather attempts.\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tl.arange(0, BX)\n    live = (ys < N)[:, None] & (xs < N)[None, :]\n\n    yc = tl.where(ys < N, ys, 0)\n    xc = tl.where(xs < N, xs, 0)\n\n    # x and y wrap; z does not (open boundary, handled by the gates below).\n    ypi = tl.where(yc + 1 >= N, 0, yc + 1)\n    ymi = tl.where(yc - 1 < 0, N - 1, yc - 1)\n    xpi = tl.where(xc + 1 >= N, 0, xc + 1)\n    xmi = tl.where(xc - 1 < 0, N - 1, xc - 1)\n\n    o_c = yc[:, None] * N + xc[None, :]\n    o_yp = ypi[:, None] * N + xc[None, :]\n    o_ym = ymi[:, None] * N + xc[None, :]\n    o_xp = yc[:, None] * N + xpi[None, :]\n    o_xm = yc[:, None] * N + xmi[None, :]\n\n    zp = tl.minimum(z + 1, LZ - 1)\n    zm = tl.maximum(z - 1, 0)\n    # `jnp.roll(...).at[:, -1].set(0.0)` in the reference: the plane past the\n    # last layer contributes zero rather than wrapping.\n    gp = tl.where(z < LZ - 1, 1.0, 0.0)\n    gm = tl.where(z > 0, 1.0, 0.0)\n\n    # ---- the site mask: inside the disk, and on the sublattice being updated.\n    disk = tl.load(disk_ptr + o_c, mask=live, other=0.0)\n    par = (xc[None, :] + yc[:, None] + z) % 2\n    upd = live & (disk > 0.5) & (par == PARITY)\n\n    # ---- per-chain parameters.\n    Jex = tl.load(par_ptr + b * 6 + 0)\n    Kan2 = tl.load(par_ptr + b * 6 + 1)\n    Hex = tl.load(par_ptr + b * 6 + 2)\n    KDM = tl.load(par_ptr + b * 6 + 3)\n    cosg = tl.load(par_ptr + b * 6 + 4)\n    sing = tl.load(par_ptr + b * 6 + 5)\n    beta = tl.load(beta_ptr)\n\n    sca = sp_ptr + b * sb\n    k1 = tl.load(k1_ptr + b * LZ * NN + z * NN + o_c, mask=live, other=0.0)\n\n    # ---- the spin at the site.\n    s0 = tl.load(sca + 0 * sc + z * NN + o_c, mask=live, other=0.0)\n    s1 = tl.load(sca + 1 * sc + z * NN + o_c, mask=live, other=0.0)\n    s2 = tl.load(sca + 2 * sc + z * NN + o_c, mask=live, other=0.0)\n\n    # ---- the trial spin: a uniform point on the sphere.\n    if HAS_RNG:\n        off = ((b * LZ + z) * N + yc[:, None]) * N + xc[None, :]\n        # One key, four distinct COUNTERS -- the textbook use of a\n        # counter-based generator. An earlier version varied the KEY instead\n        # (`sd+0..3` at a shared counter) and the engine then accepted 1.1%\n        # less than production, systematically across five temperatures,\n        # while its proposals were provably isotropic and unit-norm. That\n        # leaves the uniform: adjacent keys at one counter let `ru` correlate\n        # with the proposed direction, hence with dE, which biases the\n        # Metropolis test. Separating the counters removes the shared input.\n        sd = tl.load(step_ptr)\n        o4 = off * 4\n        g0 = tl.randn(sd, o4 + 0)\n        g1 = tl.randn(sd, o4 + 1)\n        g2 = tl.randn(sd, o4 + 2)\n        nrm = tl.sqrt(g0 * g0 + g1 * g1 + g2 * g2) + 1e-30\n        t0 = g0 / nrm * SO\n        t1 = g1 / nrm * SO\n        t2 = g2 / nrm * SO\n        ru = tl.rand(sd, o4 + 3)\n    else:\n        trb = tr_ptr + b * sb\n        t0 = tl.load(trb + 0 * sc + z * NN + o_c, mask=live, other=0.0)\n        t1 = tl.load(trb + 1 * sc + z * NN + o_c, mask=live, other=0.0)\n        t2 = tl.load(trb + 2 * sc + z * NN + o_c, mask=live, other=0.0)\n        ru = tl.load(ru_ptr + b * LZ * NN + z * NN + o_c, mask=live, other=0.0)\n\n    d0 = t0 - s0\n    d1 = t1 - s1\n    d2 = t2 - s2\n\n    # ---- the stencil, one component at a time so the six neighbour blocks of\n    # a component die before the next component\'s are loaded. Holding all 21\n    # at once spills at BY >= 8.\n    p0 = sca + 0 * sc\n    n_xp = tl.load(p0 + z * NN + o_xp, mask=live, other=0.0)\n    n_xm = tl.load(p0 + z * NN + o_xm, mask=live, other=0.0)\n    n_yp = tl.load(p0 + z * NN + o_yp, mask=live, other=0.0)\n    n_ym = tl.load(p0 + z * NN + o_ym, mask=live, other=0.0)\n    sv0 = (n_xp + n_xm + n_yp + n_ym\n           + gp * tl.load(p0 + zp * NN + o_c, mask=live, other=0.0)\n           + gm * tl.load(p0 + zm * NN + o_c, mask=live, other=0.0))\n    dyx = n_yp - n_ym                     # DMI needs this difference only\n\n    p1 = sca + 1 * sc\n    n_xp = tl.load(p1 + z * NN + o_xp, mask=live, other=0.0)\n    n_xm = tl.load(p1 + z * NN + o_xm, mask=live, other=0.0)\n    n_yp = tl.load(p1 + z * NN + o_yp, mask=live, other=0.0)\n    n_ym = tl.load(p1 + z * NN + o_ym, mask=live, other=0.0)\n    sv1 = (n_xp + n_xm + n_yp + n_ym\n           + gp * tl.load(p1 + zp * NN + o_c, mask=live, other=0.0)\n           + gm * tl.load(p1 + zm * NN + o_c, mask=live, other=0.0))\n    dxy = n_xp - n_xm\n\n    p2 = sca + 2 * sc\n    n_xp = tl.load(p2 + z * NN + o_xp, mask=live, other=0.0)\n    n_xm = tl.load(p2 + z * NN + o_xm, mask=live, other=0.0)\n    n_yp = tl.load(p2 + z * NN + o_yp, mask=live, other=0.0)\n    n_ym = tl.load(p2 + z * NN + o_ym, mask=live, other=0.0)\n    sv2 = (n_xp + n_xm + n_yp + n_ym\n           + gp * tl.load(p2 + zp * NN + o_c, mask=live, other=0.0)\n           + gm * tl.load(p2 + zm * NN + o_c, mask=live, other=0.0))\n    dxz = n_xp - n_xm\n    dyz = n_yp - n_ym\n\n    # ---- the energy difference, term for term as in `sim_core.substep`.\n    dE = -Jex * (d0 * sv0 + d1 * sv1 + d2 * sv2)\n\n    i4 = 1.0 / (SO * SO * SO * SO)\n    i6 = i4 / (SO * SO)\n    an_t = (k1 * i4 * ((t0 * t1) * (t0 * t1) + (t0 * t2) * (t0 * t2)\n                       + (t1 * t2) * (t1 * t2))\n            + Kan2 * i6 * (t0 * t1 * t2) * (t0 * t1 * t2))\n    an_s = (k1 * i4 * ((s0 * s1) * (s0 * s1) + (s0 * s2) * (s0 * s2)\n                       + (s1 * s2) * (s1 * s2))\n            + Kan2 * i6 * (s0 * s1 * s2) * (s0 * s1 * s2))\n    dE += an_t - an_s\n\n    # Interfacial DMI: in-plane bonds only, with the minus sign.\n    dE -= KDM * (d1 * dxz - d2 * dxy + d2 * dyx - d0 * dyz)\n    dE -= Hex * (d0 * cosg + d2 * sing)\n\n    acc = ((dE < 0.0) | (ru < tl.exp(-dE * beta))) & upd\n\n    ob = out_ptr + b * sb + z * NN + o_c\n    tl.store(ob + 0 * sc, tl.where(acc, t0, s0), mask=live)\n    tl.store(ob + 1 * sc, tl.where(acc, t1, s1), mask=live)\n    tl.store(ob + 2 * sc, tl.where(acc, t2, s2), mask=live)\n    if WRITE_DE:\n        tl.store(de_ptr + b * LZ * NN + z * NN + o_c, dE, mask=live)\n\n\n# ------------------------------------------------------------- JAX plumbing\n\ndef planar(x):\n    """(B, L, N, N, 3) -> (B, 3, L, N, N), paid once outside the step loop.\n\n    No explicit contiguity call is needed: `jax_triton` pins default row-major\n    layouts on its operands, so XLA has to materialise the transpose before\n    the custom call rather than passing a strided view.\n    """\n    import jax.numpy as jnp\n    return jnp.transpose(x, (0, 4, 1, 2, 3))\n\n\ndef unplanar(x):\n    import jax.numpy as jnp\n    return jnp.transpose(x, (0, 2, 3, 4, 1))\n\n\ndef pack_params(p, batch):\n    """The six scalars the kernel reads, as (B, 6).\n\n    `Kan1` is absent on purpose: `sim_core._an` takes it as an argument and\n    never uses it -- the K1 invariant is carried entirely by `K1_eff`, which\n    is already the surface-resolved per-site field.\n    """\n    import jax.numpy as jnp\n    def col(v):\n        a = jnp.asarray(v, dtype=jnp.float32).reshape(-1)\n        return jnp.broadcast_to(a, (batch,)) if a.size == 1 else a\n    g = col(p["g"])\n    return jnp.stack([col(p["Jex"]), col(p["Kan2"]), col(p["Hex"]),\n                      col(p["KDM"]), jnp.cos(g), jnp.sin(g)], axis=1)\n\n\ndef build(geo, BY=4, BX=None, num_warps=4, So=1.0):\n    """Returns `substep(spins_planar, k1, par, beta, step, parity, tr, ru)`.\n\n    `spins_planar` is component-planar. `tr`/`ru` are `None` for the in-kernel\n    RNG variant. `de` comes back only when `want_de` is set, for validation.\n    """\n    import jax\n    import jax.numpy as jnp\n    import jax_triton as jt\n\n    N, LZ = int(geo["N"]), int(geo["L"])\n    BX = int(BX) if BX else 1 << (N - 1).bit_length()\n    BY = int(BY)\n    n_strip = -(-N // BY)\n    disk = jnp.asarray(geo["disk2d"].astype(np.float32))\n\n    def substep(sp, k1, par, beta, step, parity, tr=None, ru=None,\n                want_de=False):\n        B = sp.shape[0]\n        has_rng = tr is None\n        shape = jax.ShapeDtypeStruct(sp.shape, jnp.float32)\n        # The kernel\'s argument list has to stay FIXED: `triton_call` appends\n        # one pointer per entry of `out_shape`, so dropping the dE output left\n        # `de_ptr` unbound and Triton failed to compile the signature. The\n        # output is always declared and shrinks to one element when unwanted.\n        de_shape = jax.ShapeDtypeStruct(\n            (B, LZ, N, N) if want_de else (1,), jnp.float32)\n        outs = (shape, de_shape)\n        # Likewise the trial/uniform inputs: unread under HAS_RNG, so they\n        # shrink to one element instead of allocating a full lattice of zeros\n        # on every iteration of the step loop.\n        dum = jnp.zeros((1,), jnp.float32)\n        res = jt.triton_call(\n            sp, k1, par, disk,\n            jnp.asarray(beta, jnp.float32).reshape(1),\n            jnp.asarray(step, jnp.int32).reshape(1),\n            dum if has_rng else tr,\n            dum if has_rng else ru,\n            kernel=_substep_kernel,\n            out_shape=outs,\n            grid=(B, LZ, n_strip),\n            num_warps=int(num_warps),\n            N=N, LZ=LZ, PARITY=int(parity), BY=BY, BX=BX,\n            HAS_RNG=has_rng, WRITE_DE=bool(want_de), SO=float(So),\n        )\n        return res if want_de else res[0]\n\n    substep.meta = dict(N=N, LZ=LZ, BY=BY, BX=BX, n_strip=n_strip,\n                        num_warps=int(num_warps),\n                        grid_per_replica=LZ * n_strip,\n                        lanes_per_block=BY * BX,\n                        lane_waste_pct=100.0 * (1.0 - (N * N) / float(\n                            BX * BY * n_strip)))\n    return substep\n\n\ndef sweeps(substep, k1, par, beta, n, base_step=0):\n    """`n` sweeps in one XLA program: even then odd, inside a device loop.\n\n    The loop has to live in `lax.fori_loop`. A Python loop would pay the\n    ~122 us host dispatch of a `triton_call` per substep, which is what faked\n    an earlier Triton win and reversed on amortisation.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    def body(i, sp):\n        st = base_step + 2 * i\n        sp = substep(sp, k1, par, beta, st, 0)\n        return substep(sp, k1, par, beta, st + 1, 1)\n\n    def run(sp):\n        return jax.lax.fori_loop(0, int(n), body, sp)\n    return run\n\n\ndef sweeps_extrng(substep, k1, par, beta, n, shape, base_step=0, So=1.0):\n    """`n` sweeps with the trial spins drawn by JAX, not by the kernel.\n\n    This arm exists to split the win. The in-kernel variant changes two things\n    at once -- the stencil and the random source -- so on its own it cannot\n    say which one paid. Here the stencil is the Triton one while the proposals\n    still come from JAX\'s Threefry, exactly as production draws them, so the\n    difference against production isolates the stencil and the difference\n    against the in-kernel variant isolates the RNG.\n\n    `shape` is the component-planar spin shape; drawing the normals directly\n    in that layout and normalising along the component axis is the same\n    distribution as drawing them component-minor and transposing.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    B, _, LZ, N, _ = shape\n\n    def half(sp, k, parity):\n        kt, ka = jax.random.split(k)\n        tr = jax.random.normal(kt, shape)\n        tr = tr / jnp.linalg.norm(tr, axis=1, keepdims=True) * So\n        ru = jax.random.uniform(ka, (B, LZ, N, N))\n        return substep(sp, k1, par, beta, 0, parity, tr, ru)\n\n    def body(i, v):\n        sp, k = v\n        k, k0, k1_ = jax.random.split(k, 3)\n        sp = half(sp, k0, 0)\n        return half(sp, k1_, 1), k\n\n    def run(sp):\n        return jax.lax.fori_loop(0, int(n), body,\n                                 (sp, jax.random.PRNGKey(base_step)))[0]\n    return run\n\n\n# ------------------------------------------- observables, component-planar\n#\n# These exist because of where the measurement loop spends its time. The\n# kernel works on `(B, 3, L, N, N)` while `sim_core`\'s observables expect\n# `(B, L, N, N, 3)`, and transposing once per measured sweep would copy the\n# whole lattice as often as the substep touches it -- enough to eat the win.\n# So the observables are rewritten against the component axis at position 1.\n# Every formula is the one in `sim_core`, term for term; only the axes move.\n\ndef build_obs(geo):\n    import jax.numpy as jnp\n\n    LZ, N = int(geo["L"]), int(geo["N"])\n    disco = jnp.asarray(geo["disco"].astype(np.float32))          # (L, N, N)\n    disk2d = jnp.asarray(geo["disk2d"])\n    n_act = float(geo["n_active"])\n\n    def c4(par, k):\n        """One per-chain scalar, shaped to broadcast against (B, L, N, N)."""\n        return par[:, k].reshape(-1, 1, 1, 1)\n\n    def energy(sp, par, k1):\n        # Each bond counted once, via the +x, +y, +z shifts only.\n        xp = jnp.roll(sp, -1, 4)\n        yp = jnp.roll(sp, -1, 3)\n        zp = jnp.roll(sp, -1, 2).at[:, :, -1].set(0.0)\n        E = -c4(par, 0) * jnp.sum(sp * (xp + yp + zp), axis=1)\n        sx, sy, sz = sp[:, 0], sp[:, 1], sp[:, 2]\n        E = E + (k1 * ((sx * sy) ** 2 + (sx * sz) ** 2 + (sy * sz) ** 2)\n                 + c4(par, 1) * (sx * sy * sz) ** 2)\n        E = E - c4(par, 3) * (sy * xp[:, 2] - sz * xp[:, 1]\n                              + sz * yp[:, 0] - sx * yp[:, 2])\n        E = E - c4(par, 2) * (sx * par[:, 4].reshape(-1, 1, 1, 1)\n                              + sz * par[:, 5].reshape(-1, 1, 1, 1))\n        return jnp.sum(E * disco[None], axis=(-3, -2, -1))\n\n    def mag(sp):\n        M = jnp.sum(sp * disco[None, None], axis=(2, 3, 4)) / n_act   # (B, 3)\n        return jnp.linalg.norm(M, axis=1), M[:, 2]\n\n    def _omega(a, b, c):\n        num = jnp.sum(a * jnp.cross(b, c, axis=1), axis=1)\n        den = (1.0 + jnp.sum(a * b, 1) + jnp.sum(b * c, 1)\n               + jnp.sum(c * a, 1))\n        return 2.0 * jnp.arctan2(num, den)\n\n    valid = (disk2d[:-1, :-1] & disk2d[:-1, 1:]\n             & disk2d[1:, :-1] & disk2d[1:, 1:])\n\n    def topo(sp):\n        """Berg-Luscher charge per layer: quantised, unlike a difference."""\n        s00 = sp[:, :, :, :-1, :-1]; s10 = sp[:, :, :, :-1, 1:]\n        s01 = sp[:, :, :, 1:, :-1];  s11 = sp[:, :, :, 1:, 1:]\n        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)\n        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (\n            4.0 * jnp.pi)\n\n    return dict(energy=energy, mag=mag, topo=topo, n_active=n_act)\n\n\ndef build_anneal(geo, BY=4, num_warps=4, So=1.0):\n    """The full temperature scan, with the Triton substep inside.\n\n    Structurally identical to `sim_core.build_engine`\'s `anneal`: a `lax.scan`\n    over betas, a thermalisation `fori_loop`, then a measurement `fori_loop`\n    accumulating the SAME eight quantities in the SAME order, so the result\n    feeds `sim_core.reduce_obs` unchanged.\n\n    The one difference is the random stream. `sim_core` splits a Threefry key\n    down the loop; the kernel is counter-based, so a step counter rides the\n    carry and every substep gets its own Philox offset. That makes the chain\n    reproducible from `seed` without carrying key state through the scan.\n    """\n    import jax\n    import jax.numpy as jnp\n\n    sub = build(geo, BY=BY, num_warps=num_warps, So=So)\n    O = build_obs(geo)\n    LZ = int(geo["L"])\n\n    def anneal(sp, seed, betas, par, k1, n_therm, n_meas):\n        n_therm, n_meas = int(n_therm), int(n_meas)\n\n        def pair(s, c, bt):\n            s = sub(s, k1, par, bt, c, 0)\n            return sub(s, k1, par, bt, c + 1, 1), c + 2\n\n        def per_T(carry, bt):\n            s, c = carry\n\n            def therm(_, v):\n                s, c = v\n                return pair(s, c, bt)\n            s, c = jax.lax.fori_loop(0, n_therm, therm, (s, c))\n\n            # Fluctuations are taken about a reference fixed once per\n            # temperature. Accumulating raw E and E^2 in float32 destroys the\n            # variance by cancellation -- one ulp of <E^2> can exceed the\n            # variance itself, and Cv comes out negative. A constant shift\n            # leaves the variance unchanged, so this is exact.\n            E_ref = O["energy"](s, par, k1)\n            M_ref, _ = O["mag"](s)\n            z0 = jnp.zeros(s.shape[0], jnp.float32)\n\n            def meas(_, v):\n                s, c, sE, sE2, sM, sM2, sMz, sQ = v\n                s, c = pair(s, c, bt)\n                dE = O["energy"](s, par, k1) - E_ref\n                M, Mz = O["mag"](s)\n                dM = M - M_ref\n                Q = O["topo"](s)[:, LZ // 2]\n                return (s, c, sE + dE, sE2 + dE * dE, sM + dM,\n                        sM2 + dM * dM, sMz + Mz, sQ + Q)\n\n            s, c, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(\n                0, n_meas, meas, (s, c, z0, z0, z0, z0, z0, z0))\n            out = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            # Replica 0, middle layer, z component -- the same snapshot\n            # `sim_core` takes, read off the planar layout.\n            snap = s[0, 2, LZ // 2]\n            return (s, c), (out, snap)\n\n        c0 = jnp.asarray(int(seed), jnp.int32) * jnp.int32(1000003)\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (sp, c0), betas)\n        return all_obs, s, snaps\n\n    anneal.meta = dict(sub.meta)\n    return anneal\n\n\n# ------------------------------------------- the observables, also in Triton\n#\n# The ladder measurement put a number on what the substep could NOT reach:\n# the bare substep ran 66.6% faster, the full ladder only 59.3%, and those\n# seven points are the measurement loop, which recomputes the energy, the\n# magnetisation and the topological charge in JAX on every measured sweep.\n# These two kernels are that ceiling.\n#\n# Both emit PER-BLOCK PARTIALS rather than accumulating with atomics. Atomic\n# float adds would make the result depend on block completion order, so two\n# runs of the same state would disagree in the last bits and the exactness\n# test below would have nothing to stand on. The partial array is tiny\n# (B x LZ*n_strip x 4), so the final sum costs nothing in JAX.\n\n@triton.jit\ndef _eM_kernel(\n    sp_ptr,          # (B, 3, LZ, N, N)  f32\n    k1_ptr,          # (B, LZ, N, N)     f32\n    par_ptr,         # (B, 6)            f32\n    disk_ptr,        # (N, N)            f32\n    out_ptr,         # (B, LZ*n_strip, 4) f32  -> E, Mx, My, Mz\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    BY: tl.constexpr,\n    BX: tl.constexpr,\n    NSTRIP: tl.constexpr,\n    SO: tl.constexpr,\n):\n    b = tl.program_id(0)\n    z = tl.program_id(1)\n    ty = tl.program_id(2)\n\n    NN = N * N\n    sb = 3 * LZ * NN\n    sc = LZ * NN\n\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tl.arange(0, BX)\n    live = (ys < N)[:, None] & (xs < N)[None, :]\n    yc = tl.where(ys < N, ys, 0)\n    xc = tl.where(xs < N, xs, 0)\n\n    # Each bond counted once: the +x, +y and +z shifts only.\n    ypi = tl.where(yc + 1 >= N, 0, yc + 1)\n    xpi = tl.where(xc + 1 >= N, 0, xc + 1)\n    o_c = yc[:, None] * N + xc[None, :]\n    o_yp = ypi[:, None] * N + xc[None, :]\n    o_xp = yc[:, None] * N + xpi[None, :]\n    zp = tl.minimum(z + 1, LZ - 1)\n    gp = tl.where(z < LZ - 1, 1.0, 0.0)\n\n    msk = live & (tl.load(disk_ptr + o_c, mask=live, other=0.0) > 0.5)\n    w = tl.where(msk, 1.0, 0.0)\n\n    Jex = tl.load(par_ptr + b * 6 + 0)\n    Kan2 = tl.load(par_ptr + b * 6 + 1)\n    Hex = tl.load(par_ptr + b * 6 + 2)\n    KDM = tl.load(par_ptr + b * 6 + 3)\n    cosg = tl.load(par_ptr + b * 6 + 4)\n    sing = tl.load(par_ptr + b * 6 + 5)\n\n    sca = sp_ptr + b * sb\n    k1 = tl.load(k1_ptr + b * LZ * NN + z * NN + o_c, mask=live, other=0.0)\n\n    p0 = sca + 0 * sc; p1 = sca + 1 * sc; p2 = sca + 2 * sc\n    s0 = tl.load(p0 + z * NN + o_c, mask=live, other=0.0)\n    s1 = tl.load(p1 + z * NN + o_c, mask=live, other=0.0)\n    s2 = tl.load(p2 + z * NN + o_c, mask=live, other=0.0)\n    x0 = tl.load(p0 + z * NN + o_xp, mask=live, other=0.0)\n    x1 = tl.load(p1 + z * NN + o_xp, mask=live, other=0.0)\n    x2 = tl.load(p2 + z * NN + o_xp, mask=live, other=0.0)\n    y0 = tl.load(p0 + z * NN + o_yp, mask=live, other=0.0)\n    y1 = tl.load(p1 + z * NN + o_yp, mask=live, other=0.0)\n    y2 = tl.load(p2 + z * NN + o_yp, mask=live, other=0.0)\n    z0 = gp * tl.load(p0 + zp * NN + o_c, mask=live, other=0.0)\n    z1 = gp * tl.load(p1 + zp * NN + o_c, mask=live, other=0.0)\n    z2 = gp * tl.load(p2 + zp * NN + o_c, mask=live, other=0.0)\n\n    E = -Jex * (s0 * (x0 + y0 + z0) + s1 * (x1 + y1 + z1)\n                + s2 * (x2 + y2 + z2))\n    i4 = 1.0 / (SO * SO * SO * SO)\n    i6 = i4 / (SO * SO)\n    E += (k1 * i4 * ((s0 * s1) * (s0 * s1) + (s0 * s2) * (s0 * s2)\n                     + (s1 * s2) * (s1 * s2))\n          + Kan2 * i6 * (s0 * s1 * s2) * (s0 * s1 * s2))\n    E -= KDM * (s1 * x2 - s2 * x1 + s2 * y0 - s0 * y2)\n    E -= Hex * (s0 * cosg + s2 * sing)\n\n    ob = out_ptr + b * (LZ * NSTRIP) * 4 + (z * NSTRIP + ty) * 4\n    tl.store(ob + 0, tl.sum(E * w))\n    tl.store(ob + 1, tl.sum(s0 * w))\n    tl.store(ob + 2, tl.sum(s1 * w))\n    tl.store(ob + 3, tl.sum(s2 * w))\n\n\n@triton.jit\ndef _topo_kernel(\n    sp_ptr,          # (B, 3, LZ, N, N)  f32\n    out_ptr,         # (B, 4, N-1, N-1)  f32  -> n1, d1, n2, d2\n    N: tl.constexpr,\n    LZ: tl.constexpr,\n    ZMID: tl.constexpr,\n    BY: tl.constexpr,\n    BX: tl.constexpr,\n):\n    """The plaquette terms of the Berg-Luscher charge, middle layer only.\n\n    It emits the four arguments of the two `atan2` rather than the angle:\n    Triton 3.8 has no `atan2` outside libdevice, and the branch matters here\n    because the denominator `1 + a.b + b.c + c.a` can go negative on a\n    frustrated plaquette, so `atan(n/d)` is not a substitute. The twelve\n    loads and the cross products -- the expensive part -- stay in the kernel\n    and JAX finishes with one cheap elementwise pass over 38x38.\n\n    One z plane, because that is the only layer `anneal` reads, so the\n    plaquettes are not evaluated five times over.\n    """\n    b = tl.program_id(0)\n    ty = tl.program_id(1)\n    tx = tl.program_id(2)\n\n    NN = N * N\n    sb = 3 * LZ * NN\n    sc = LZ * NN\n    M = N - 1                      # the plaquette grid\n\n    ys = ty * BY + tl.arange(0, BY)\n    xs = tx * BX + tl.arange(0, BX)\n    # Spans y..y+1 and x..x+1 with NO wrap, matching `spins[..., :-1, :-1, :]`.\n    live = (ys < M)[:, None] & (xs < M)[None, :]\n    yc = tl.where(ys < M, ys, 0)\n    xc = tl.where(xs < M, xs, 0)\n\n    o00 = yc[:, None] * N + xc[None, :]\n    o10 = yc[:, None] * N + (xc + 1)[None, :]\n    o01 = (yc + 1)[:, None] * N + xc[None, :]\n    o11 = (yc + 1)[:, None] * N + (xc + 1)[None, :]\n\n    base = sp_ptr + b * sb + ZMID * NN\n    a0 = tl.load(base + 0 * sc + o00, mask=live, other=0.0)\n    a1 = tl.load(base + 1 * sc + o00, mask=live, other=0.0)\n    a2 = tl.load(base + 2 * sc + o00, mask=live, other=0.0)\n    u0 = tl.load(base + 0 * sc + o10, mask=live, other=0.0)\n    u1 = tl.load(base + 1 * sc + o10, mask=live, other=0.0)\n    u2 = tl.load(base + 2 * sc + o10, mask=live, other=0.0)\n    c0 = tl.load(base + 0 * sc + o01, mask=live, other=0.0)\n    c1 = tl.load(base + 1 * sc + o01, mask=live, other=0.0)\n    c2 = tl.load(base + 2 * sc + o01, mask=live, other=0.0)\n    e0 = tl.load(base + 0 * sc + o11, mask=live, other=0.0)\n    e1 = tl.load(base + 1 * sc + o11, mask=live, other=0.0)\n    e2 = tl.load(base + 2 * sc + o11, mask=live, other=0.0)\n\n    # omega(a, u, v) = 2 atan2( a . (u x v), 1 + a.u + u.v + v.a )\n    n1 = (a0 * (u1 * e2 - u2 * e1) + a1 * (u2 * e0 - u0 * e2)\n          + a2 * (u0 * e1 - u1 * e0))\n    d1 = (1.0 + (a0 * u0 + a1 * u1 + a2 * u2)\n          + (u0 * e0 + u1 * e1 + u2 * e2)\n          + (e0 * a0 + e1 * a1 + e2 * a2))\n    n2 = (a0 * (e1 * c2 - e2 * c1) + a1 * (e2 * c0 - e0 * c2)\n          + a2 * (e0 * c1 - e1 * c0))\n    d2 = (1.0 + (a0 * e0 + a1 * e1 + a2 * e2)\n          + (e0 * c0 + e1 * c1 + e2 * c2)\n          + (c0 * a0 + c1 * a1 + c2 * a2))\n\n    off = yc[:, None] * M + xc[None, :]\n    ob = out_ptr + b * 4 * M * M\n    tl.store(ob + 0 * M * M + off, n1, mask=live)\n    tl.store(ob + 1 * M * M + off, d1, mask=live)\n    tl.store(ob + 2 * M * M + off, n2, mask=live)\n    tl.store(ob + 3 * M * M + off, d2, mask=live)\n\n\ndef build_obs_triton(geo, BY=4, BX=None, num_warps=4, So=1.0):\n    """E, M and Q from two Triton kernels plus one tiny JAX reduction."""\n    import jax\n    import jax.numpy as jnp\n    import jax_triton as jt\n\n    N, LZ = int(geo["N"]), int(geo["L"])\n    BX = int(BX) if BX else 1 << (N - 1).bit_length()\n    BY = int(BY)\n    nst = -(-N // BY)\n    M = N - 1\n    nqy, nqx = -(-M // BY), -(-M // BX)\n    disk = jnp.asarray(geo["disk2d"].astype(np.float32))\n    d2d = geo["disk2d"]\n    qvalid = jnp.asarray((d2d[:-1, :-1] & d2d[:-1, 1:]\n                          & d2d[1:, :-1] & d2d[1:, 1:]))\n    n_act = float(geo["n_active"])\n    four_pi = 4.0 * float(np.pi)\n\n    def eM(sp, par, k1):\n        B = sp.shape[0]\n        part = jt.triton_call(\n            sp, k1, par, disk,\n            kernel=_eM_kernel,\n            out_shape=jax.ShapeDtypeStruct((B, LZ * nst, 4), jnp.float32),\n            grid=(B, LZ, nst),\n            num_warps=int(num_warps),\n            N=N, LZ=LZ, BY=BY, BX=BX, NSTRIP=nst, SO=float(So))\n        tot = jnp.sum(part, axis=1)\n        M = tot[:, 1:4] / n_act\n        return tot[:, 0], jnp.linalg.norm(M, axis=1), M[:, 2]\n\n    def topo_mid(sp):\n        B = sp.shape[0]\n        t = jt.triton_call(\n            sp,\n            kernel=_topo_kernel,\n            out_shape=jax.ShapeDtypeStruct((B, 4, M, M), jnp.float32),\n            grid=(B, nqy, nqx),\n            num_warps=int(num_warps),\n            N=N, LZ=LZ, ZMID=LZ // 2, BY=BY, BX=BX)\n        w = 2.0 * (jnp.arctan2(t[:, 0], t[:, 1])\n                   + jnp.arctan2(t[:, 2], t[:, 3]))\n        return jnp.sum(jnp.where(qvalid, w, 0.0), axis=(1, 2)) / four_pi\n\n    return dict(eM=eM, topo_mid=topo_mid, n_active=n_act,\n                meta=dict(BY=BY, BX=BX, n_strip=nst, q_grid=(nqy, nqx)))\n\n\ndef build_anneal_full(geo, BY=4, num_warps=4, So=1.0):\n    """The anneal with BOTH the substep and the observables in Triton."""\n    import jax\n    import jax.numpy as jnp\n\n    sub = build(geo, BY=BY, num_warps=num_warps, So=So)\n    O = build_obs_triton(geo, BY=BY, num_warps=num_warps, So=So)\n    LZ = int(geo["L"])\n\n    def anneal(sp, seed, betas, par, k1, n_therm, n_meas):\n        n_therm, n_meas = int(n_therm), int(n_meas)\n\n        def pair(s, c, bt):\n            s = sub(s, k1, par, bt, c, 0)\n            return sub(s, k1, par, bt, c + 1, 1), c + 2\n\n        def per_T(carry, bt):\n            s, c = carry\n\n            def therm(_, v):\n                s, c = v\n                return pair(s, c, bt)\n            s, c = jax.lax.fori_loop(0, n_therm, therm, (s, c))\n\n            E_ref, M_ref, _ = O["eM"](s, par, k1)\n            z0 = jnp.zeros(s.shape[0], jnp.float32)\n\n            def meas(_, v):\n                s, c, sE, sE2, sM, sM2, sMz, sQ = v\n                s, c = pair(s, c, bt)\n                E, M, Mz = O["eM"](s, par, k1)\n                dE = E - E_ref\n                dM = M - M_ref\n                Q = O["topo_mid"](s)\n                return (s, c, sE + dE, sE2 + dE * dE, sM + dM,\n                        sM2 + dM * dM, sMz + Mz, sQ + Q)\n\n            s, c, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(\n                0, n_meas, meas, (s, c, z0, z0, z0, z0, z0, z0))\n            out = jnp.stack([sE / n_meas, sE2 / n_meas,\n                             sM / n_meas, sM2 / n_meas,\n                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])\n            return (s, c), (out, s[0, 2, LZ // 2])\n\n        c0 = jnp.asarray(int(seed), jnp.int32) * jnp.int32(1000003)\n        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (sp, c0), betas)\n        return all_obs, s, snaps\n\n    anneal.meta = dict(sub.meta, **O["meta"])\n    return anneal\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "1d36dd32e4bddfa299b91b89f305165b5598cae8c8ba4932ab9fdeeef1b4b862", "triton_halo.py alterado"
open("triton_halo.py", "w").write(_src)
print("triton_halo.py  {} bytes  sha256 ok".format(len(_src)))


In [ ]:
# triton_bench.py -- byte-identical to the repository copy. sha256 1579738215b7dcc2b70ef5f193402ec0d6fd2f939087fddd51e474dc4bd00ed9
import hashlib
_src = '"""Does the hand-written kernel travel? One self-contained benchmark.\n\nThe whole Triton result was measured on H100s rented by the hour. That leaves\ntwo things unknown. Whether the kernel even COMPILES elsewhere -- Triton wants\ncompute capability 7.0 or better, so Kaggle\'s P100 (6.0) should refuse it\nwhile its T4 (7.5) should not. And whether the gain is an H100 property: the\nH100 has 132 SMs and enormous L2, and a stencil that lives in registers might\nbehave very differently on a 40-SM T4.\n\nSo this module runs, in order:\n\n  1. the environment, including compute capability, because a refusal there is\n     a result and not a failure;\n  2. exactness -- the stencil against `sim_core.substep` on IDENTICAL\n     randomness, which must give zero differing spins;\n  3. physics -- both engines to equilibrium with their own randomness, whose\n     energies must agree within the spread the run itself reports;\n  4. the substep rate, production against the kernel, over a grid of layer\n     counts and batches, which also scores the width rule SEPARATELY for each\n     engine;\n  5. a short ladder, so the figure quoted is the full anneal and not a step.\n\nNothing is timed before the physics agrees, and every arm is interleaved so a\nGPU that warms up spreads its drift across both.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport time\n\nimport numpy as np\n\n\ndef _v2(n):\n    return (n & -n).bit_length() - 1\n\n\ndef environment():\n    import subprocess\n    import sys\n    out = {}\n    try:\n        out["smi"] = subprocess.run(\n            ["nvidia-smi", "--query-gpu=name,memory.total,clocks.max.sm,"\n             "driver_version", "--format=csv,noheader"],\n            capture_output=True, text=True).stdout.strip()\n    except Exception as e:\n        out["smi"] = f"nvidia-smi no disponible: {e}"\n    try:\n        import jax\n        out["jax"] = jax.__version__\n        out["devices"] = [str(d) for d in jax.devices()]\n        d = jax.devices()[0]\n        # Compute capability decides whether Triton will emit anything at all.\n        cc = getattr(d, "compute_capability", None)\n        out["compute_capability"] = str(cc) if cc else "desconocida"\n        out["device_kind"] = getattr(d, "device_kind", "?")\n    except Exception as e:\n        out["jax_error"] = f"{type(e).__name__}: {e}"\n    for mod in ("triton", "jax_triton"):\n        try:\n            m = __import__(mod)\n            out[mod] = getattr(m, "__version__", "?")\n        except Exception as e:\n            out[f"{mod}_error"] = f"{type(e).__name__}: {e}"\n    out["python"] = sys.version.split()[0]\n    return out\n\n\ndef run(batches=(64, 65, 100, 101), layers=(5,), repeats=5, steps=200,\n        Rd=18.30, by=4, warps=4, replicas=50, n_temps=20, n_therm=300,\n        n_meas=150, seed=11, out=None, label="kaggle"):\n    import jax\n    import jax.numpy as jnp\n    import sim_core as S\n\n    res = dict(label=label, env=environment(), rows=[], ok=False)\n    print("=== entorno ===", flush=True)\n    for k in ("smi", "device_kind", "compute_capability", "jax", "triton",\n              "jax_triton", "python"):\n        if k in res["env"]:\n            print(f"  {k:<20} {res[\'env\'][k]}", flush=True)\n    cc = res["env"].get("compute_capability", "")\n    if cc and cc.replace(".", "").isdigit() and float(\n            cc if "." in cc else f"{cc[0]}.{cc[1:]}") < 7.0:\n        print(f"\\n  AVISO: capacidad de computo {cc} < 7.0. Triton exige 7.0+,"\n              f" asi que lo mas probable es que se niegue a compilar. Eso es"\n              f" un resultado, no un fallo: elige el acelerador T4 (7.5).",\n              flush=True)\n\n    try:\n        import triton_halo as TH\n    except Exception as e:\n        res["error"] = f"{type(e).__name__}: {e}"\n        print(f"\\nTRITON NO DISPONIBLE: {res[\'error\']}", flush=True)\n        if out:\n            json.dump(res, open(out, "w"), indent=1)\n        return res\n\n    kB = 0.086173404\n    beta = 1.0 / (kB * 5.0)\n    K = int(steps)\n    P0 = dict(Jex=1.0, Kan1=0.177, Kan2=0.0, KanS=0.139, Hex=0.483,\n              KDM=0.880, g=float(np.deg2rad(90.0)))\n\n    def params(b):\n        c = lambda v: jnp.full((b, 1, 1, 1), float(v), jnp.float32)\n        return {k: c(v) for k, v in P0.items()}\n\n    # ---------------------------------------------------- 2. exactness\n    print("\\n=== exactitud: el stencil contra sim_core.substep ===",\n          flush=True)\n    geo = S.build_geometry(Rd=float(Rd), L=int(layers[0]))\n    eng = S.build_engine(geo)\n    surf = jnp.asarray(geo["surface"][None, ...])\n    ev = jnp.asarray(geo["even"][None, ..., None])\n    od = jnp.asarray(geo["odd"][None, ..., None])\n    Bv = 8\n    P = params(Bv)\n    k1 = jnp.where(surf, P["KanS"], P["Kan1"])\n    sp0 = S.init_spins(jax.random.PRNGKey(3), geo, Bv)\n    kt, ka = jax.random.split(jax.random.PRNGKey(101))\n    tr = jax.random.normal(kt, sp0.shape)\n    tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True)\n    ru = jax.random.uniform(ka, sp0.shape[:-1])\n    sub = TH.build(geo, BY=int(by), num_warps=int(warps))\n    par = TH.pack_params(P, Bv)\n    spP = TH.planar(sp0)\n    trP = TH.planar(tr)\n    ex = []\n    for parity, mask, nm in ((0, ev, "par"), (1, od, "impar")):\n        # `substep` draws its own randomness, so the reference is rebuilt here\n        # from the SAME tr/ru the kernel is given -- otherwise this compares\n        # two different chains and proves nothing.\n        trm = jnp.where(mask, tr, sp0)\n        dS = trm - sp0\n        xp = jnp.roll(sp0, -1, 3); xm = jnp.roll(sp0, 1, 3)\n        yp = jnp.roll(sp0, -1, 2); ym = jnp.roll(sp0, 1, 2)\n        zp = jnp.roll(sp0, -1, 1).at[:, -1].set(0.0)\n        zm = jnp.roll(sp0, 1, 1).at[:, 0].set(0.0)\n        dE = -P["Jex"] * jnp.sum(dS * (xp + xm + yp + ym + zp + zm), -1)\n        an = lambda s: (k1 * ((s[..., 0] * s[..., 1]) ** 2\n                              + (s[..., 0] * s[..., 2]) ** 2\n                              + (s[..., 1] * s[..., 2]) ** 2)\n                        + P["Kan2"] * (s[..., 0] * s[..., 1] * s[..., 2]) ** 2)\n        dE = dE + an(trm) - an(sp0)\n        dE = dE - P["KDM"] * (dS[..., 1] * (xp[..., 2] - xm[..., 2])\n                              - dS[..., 2] * (xp[..., 1] - xm[..., 1])\n                              + dS[..., 2] * (yp[..., 0] - ym[..., 0])\n                              - dS[..., 0] * (yp[..., 2] - ym[..., 2]))\n        dE = dE - P["Hex"] * (dS[..., 0] * jnp.cos(P["g"])\n                              + dS[..., 2] * jnp.sin(P["g"]))\n        acc = ((dE < 0) | (ru < jnp.exp(-dE * beta)))[..., None] & mask\n        ref = jnp.where(acc, trm, sp0)\n        got, dE_t = sub(spP, k1, par, beta, 0, parity, trP, ru, want_de=True)\n        sel = np.broadcast_to(np.asarray(mask[..., 0]), dE.shape)\n        mad = float(np.max(np.abs(np.asarray(dE)[sel]\n                                  - np.asarray(dE_t)[sel])))\n        bad = int(np.sum(np.asarray(ref) != np.asarray(TH.unplanar(got))))\n        ex.append(dict(sublattice=nm, dE_max_abs=mad, spin_mismatch=bad,\n                       spin_total=int(np.asarray(ref).size)))\n        print(f"  {nm:>5}: dE max |dif| {mad:.3e}   espines distintos "\n              f"{bad}/{np.asarray(ref).size}", flush=True)\n    res["exactness"] = ex\n    res["exact_ok"] = all(r["spin_mismatch"] == 0 for r in ex)\n\n    # ------------------------------------------------------ 3. physics\n    print("\\n=== fisica: las dos cadenas a equilibrio, cada una con su RNG ===",\n          flush=True)\n    Bp = 16\n    Pp = params(Bp)\n    k1p = jnp.where(surf, Pp["KanS"], Pp["Kan1"])\n    parp = TH.pack_params(Pp, Bp)\n    spE = S.init_spins(jax.random.PRNGKey(11), geo, Bp)\n    subp = TH.build(geo, BY=int(by), num_warps=int(warps))\n    n_act = float(geo["n_active"])\n    ph = []\n    for T in (20.0, 5.0, 1.0):\n        bT = 1.0 / (kB * T)\n        n_eq = 400\n\n        def jx(s):\n            def body(i, v):\n                st, k = v\n                k, a, b = jax.random.split(k, 3)\n                st = eng["substep"](st, a, bT, ev, Pp, k1p)\n                st = eng["substep"](st, b, bT, od, Pp, k1p)\n                return st, k\n            return jax.lax.fori_loop(0, n_eq, body,\n                                     (s, jax.random.PRNGKey(77)))[0]\n\n        sj = jax.block_until_ready(jax.jit(jx)(spE))\n        st = TH.unplanar(jax.block_until_ready(jax.jit(\n            TH.sweeps(subp, k1p, parp, bT, n_eq))(TH.planar(spE))))\n        ej = float(jnp.mean(eng["energy"](sj, Pp, k1p)) / n_act)\n        et = float(jnp.mean(eng["energy"](st, Pp, k1p)) / n_act)\n        sd = float(jnp.std(eng["energy"](sj, Pp, k1p)) / n_act)\n        row = dict(T=T, E_prod=ej, E_halo=et, E_sd=sd,\n                   sigma=abs(et - ej) / (sd + 1e-30),\n                   M_prod=float(jnp.mean(eng["magnetization"](sj)[0])),\n                   M_halo=float(jnp.mean(eng["magnetization"](st)[0])))\n        ph.append(row)\n        print(f"  T={T:>5.1f}  E/sitio {ej:+9.5f} / {et:+9.5f}  "\n              f"({row[\'sigma\']:.2f} sigma)   |M| {row[\'M_prod\']:.4f} / "\n              f"{row[\'M_halo\']:.4f}", flush=True)\n    res["physics"] = ph\n    res["physics_ok"] = all(r["sigma"] < 3.0 for r in ph)\n    if not (res["exact_ok"] and res["physics_ok"]):\n        print("\\n  AVISO: la fisica NO coincide. Los tiempos de abajo no son "\n              "comparables.", flush=True)\n\n    # ------------------------------- 4. the substep rate, and the width rule\n    print(f"\\n=== ritmo del paso, {K} barridos por llamada, "\n          f"{repeats} repeticiones intercaladas ===", flush=True)\n    print(f"{\'L\':>3}{\'B\':>6}{\'v2(B)+v2(L)\':>13}{\'produccion\':>13}"\n          f"{\'kernel\':>10}{\'cambio\':>9}", flush=True)\n    for L in layers:\n        g = S.build_geometry(Rd=float(Rd), L=int(L))\n        e2 = S.build_engine(g)\n        s2 = jnp.asarray(g["surface"][None, ...])\n        v2e = jnp.asarray(g["even"][None, ..., None])\n        v2o = jnp.asarray(g["odd"][None, ..., None])\n        for B in batches:\n            Pb = params(B)\n            k1b = jnp.where(s2, Pb["KanS"], Pb["Kan1"])\n            parb = TH.pack_params(Pb, B)\n            spb = S.init_spins(jax.random.PRNGKey(seed + B + L), g, B)\n            spbP = TH.planar(spb)\n            sb = TH.build(g, BY=int(by), num_warps=int(warps))\n\n            def prod(s):\n                def body(i, v):\n                    st, k = v\n                    k, a, b = jax.random.split(k, 3)\n                    st = e2["substep"](st, a, beta, v2e, Pb, k1b)\n                    st = e2["substep"](st, b, beta, v2o, Pb, k1b)\n                    return st, k\n                return jax.lax.fori_loop(0, K, body,\n                                         (s, jax.random.PRNGKey(0)))[0]\n\n            f1 = jax.jit(prod).lower(spb).compile()\n            f2 = jax.jit(TH.sweeps(sb, k1b, parb, beta, K)\n                         ).lower(spbP).compile()\n            jax.block_until_ready(f1(spb)); jax.block_until_ready(f2(spbP))\n            a1, a2 = [], []\n            for _ in range(int(repeats)):\n                t = time.perf_counter(); jax.block_until_ready(f1(spb))\n                a1.append(time.perf_counter() - t)\n                t = time.perf_counter(); jax.block_until_ready(f2(spbP))\n                a2.append(time.perf_counter() - t)\n            m1, m2 = float(np.median(a1)), float(np.median(a2))\n            row = dict(L=int(L), B=int(B), suma=_v2(L) + _v2(B),\n                       prod_us=1e6 * m1 / (K * B),\n                       halo_us=1e6 * m2 / (K * B),\n                       change_pct=100 * (m2 / m1 - 1))\n            res["rows"].append(row)\n            print(f"{L:>3}{B:>6}{row[\'suma\']:>13}{row[\'prod_us\']:>13.3f}"\n                  f"{row[\'halo_us\']:>10.3f}{row[\'change_pct\']:>+8.1f}%",\n                  flush=True)\n\n    pen = {}\n    for L in layers:\n        rs = [r for r in res["rows"] if r["L"] == L]\n        hit = [r for r in rs if r["suma"] >= 2]\n        miss = [r for r in rs if r["suma"] < 2]\n        if hit and miss:\n            for key, lab in (("prod_us", "produccion"), ("halo_us", "kernel")):\n                p = 100 * (float(np.mean([r[key] for r in hit]))\n                           / float(np.mean([r[key] for r in miss])) - 1)\n                pen[f"L{L}_{lab}"] = p\n    if pen:\n        print(f"\\n=== la regla del ancho: castigo por v2(B)+v2(L) >= 2 ===",\n              flush=True)\n        for k, v in pen.items():\n            print(f"  {k:<22}{v:+7.1f}%", flush=True)\n    res["penalty"] = pen\n\n    # ------------------------------------------------- 5. the short ladder\n    print(f"\\n=== escalera corta: {n_temps}T x {n_therm}+{n_meas}, "\n          f"{replicas} replicas ===", flush=True)\n    Bl = int(replicas)\n    Pl = params(Bl)\n    k1l = jnp.where(surf, Pl["KanS"], Pl["Kan1"])\n    parl = TH.pack_params(Pl, Bl)\n    Tl = np.linspace(20.0, 0.1, int(n_temps))\n    btl = jnp.asarray(1.0 / (kB * Tl), jnp.float32)\n    spl = S.init_spins(jax.random.PRNGKey(5), geo, Bl)\n    annF = TH.build_anneal_full(geo, BY=int(by), num_warps=int(warps))\n    tot = int(n_temps) * (int(n_therm) + int(n_meas))\n    lad = {}\n    for nm, fn, a0 in (\n            ("produccion", lambda s: eng["anneal"](\n                s, jax.random.PRNGKey(42), btl, Pl, int(n_therm),\n                int(n_meas)), spl),\n            ("kernel", lambda s: annF(s, 42, btl, parl, k1l, int(n_therm),\n                                      int(n_meas)), TH.planar(spl))):\n        f = jax.jit(fn).lower(a0).compile()\n        t0 = time.perf_counter()\n        o = jax.block_until_ready(f(a0))\n        dt = time.perf_counter() - t0\n        lad[nm] = dict(wall_s=dt, us_per_replica_sweep=1e6 * dt / (tot * Bl))\n        print(f"  {nm:>11}: {dt:7.2f} s   "\n              f"{lad[nm][\'us_per_replica_sweep\']:7.3f} us/replica-barrido",\n              flush=True)\n        del o\n    lad["change_pct"] = 100 * (lad["kernel"]["wall_s"]\n                               / lad["produccion"]["wall_s"] - 1)\n    res["ladder"] = dict(n_temps=int(n_temps), n_therm=int(n_therm),\n                         n_meas=int(n_meas), replicas=Bl, total_sweeps=tot,\n                         arms=lad)\n    print(f"\\n  la escalera completa: {lad[\'change_pct\']:+.1f}%", flush=True)\n    res["ok"] = bool(res["exact_ok"] and res["physics_ok"])\n    if out:\n        json.dump(res, open(out, "w"), indent=1)\n        print(f"\\n  guardado {out}", flush=True)\n    return res\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "1579738215b7dcc2b70ef5f193402ec0d6fd2f939087fddd51e474dc4bd00ed9", "triton_bench.py alterado"
open("triton_bench.py", "w").write(_src)
print("triton_bench.py  {} bytes  sha256 ok".format(len(_src)))


## La medición

La rejilla es `L ∈ {5, 6} × B ∈ {64, 65, 100, 101}`. Esos cuatro lotes están
elegidos para que la regla del ancho quede puntuada dentro de cada capa:
`v₂(64)=6` y `v₂(100)=2` la disparan, `v₂(65)=v₂(101)=0` no. En la H100 el
castigo fue **+17.8 %** para producción y **+0.3 %** para el kernel.


In [ ]:
import sys, json
sys.path.insert(0, ".")
import triton_bench

res = triton_bench.run(
    batches=(64, 65, 100, 101), layers=(5, 6),
    repeats=5, steps=200, Rd=18.30, by=4, warps=4,
    replicas=50, n_temps=20, n_therm=300, n_meas=150,
    out="/kaggle/working/triton_bench_kaggle.json", label="kaggle")

## Comparación contra la H100

La fila de la H100 son los números medidos allí con la misma rejilla. Lo que
se compara **no** es el tiempo absoluto —una T4 es mucho más lenta y eso no
dice nada— sino **la ganancia relativa** y **si la regla del ancho sigue
muerta** en el kernel.


In [ ]:
H100 = {"paso_cambio_pct": -66.6, "escalera_cambio_pct": -62.8,
        "castigo_produccion_pct": 17.8, "castigo_kernel_pct": 0.3}

if not res.get("ok"):
    print("La fisica no coincidio o Triton no compilo: no hay nada que "
          "comparar.")
    print("  error:", res.get("error", "ver avisos arriba"))
else:
    import statistics as st
    ch = st.mean([r["change_pct"] for r in res["rows"]])
    lad = res["ladder"]["arms"]["change_pct"]
    pk = [v for k, v in res["penalty"].items() if "kernel" in k]
    pp = [v for k, v in res["penalty"].items() if "produccion" in k]
    print(f"{'':<34}{'esta GPU':>12}{'H100':>10}")
    print(f"{'paso, kernel vs produccion':<34}{ch:>11.1f}%"
          f"{H100['paso_cambio_pct']:>9.1f}%")
    print(f"{'escalera completa':<34}{lad:>11.1f}%"
          f"{H100['escalera_cambio_pct']:>9.1f}%")
    if pp:
        print(f"{'castigo del ancho, produccion':<34}"
              f"{st.mean(pp):>+11.1f}%{H100['castigo_produccion_pct']:>+9.1f}%")
    if pk:
        print(f"{'castigo del ancho, kernel':<34}"
              f"{st.mean(pk):>+11.1f}%{H100['castigo_kernel_pct']:>+9.1f}%")
    print()
    print("Lectura: si la ganancia se sostiene y el castigo del kernel sigue")
    print("cerca de cero, el resultado NO es una propiedad de la H100.")